# Patient classification basics

Predict the recorded in/out label from the patient CSV. This is a classification exercise, not a treatment recommendation.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

root = Path.cwd()
if not (root / "patient_data.csv").exists():
    root = root.parent
data = pd.read_csv(root / "patient_data.csv")
print(data.shape)
data.head()


In [ ]:
print(data.dtypes)
print(data["SOURCE"].value_counts())


In [ ]:
print(data.isna().sum())
data.describe()


In [ ]:
X = data.drop(columns="SOURCE")
y = data["SOURCE"]
print("Features:", X.columns.tolist())


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Training rows:", len(X_train), "Test rows:", len(X_test))


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_columns = X.select_dtypes(include="number").columns.tolist()
numeric_steps = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
category_steps = OneHotEncoder(handle_unknown="ignore")
preprocess = ColumnTransformer([
    ("numbers", numeric_steps, numeric_columns),
    ("sex", category_steps, ["SEX"]),
])


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model = make_pipeline(preprocess, LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)
prediction = model.predict(X_test)
print("Logistic regression test accuracy:", accuracy_score(y_test, prediction))


In [ ]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier

forest = make_pipeline(
    clone(preprocess),
    RandomForestClassifier(n_estimators=80, max_depth=6, random_state=42, n_jobs=1),
)
forest.fit(X_train, y_train)
forest_prediction = forest.predict(X_test)
print("Random forest test accuracy:", accuracy_score(y_test, forest_prediction))


In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_prediction = baseline.predict(X_test)
print("Majority baseline accuracy:", accuracy_score(y_test, baseline_prediction))


In [ ]:
scores = pd.DataFrame({
    "model": ["majority baseline", "logistic regression", "random forest"],
    "test_accuracy": [
        accuracy_score(y_test, baseline_prediction),
        accuracy_score(y_test, prediction),
        accuracy_score(y_test, forest_prediction),
    ],
})
print(scores.to_string(index=False))


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, prediction, labels=["in", "out"], zero_division=0))


In [ ]:
# Set this to True to save the small score table. No patient records are exported.
save_scores = False
if save_scores:
    (root / "results").mkdir(exist_ok=True)
    scores.to_csv(root / "results/patient_scores.csv", index=False)


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(y_test, prediction, labels=['in', 'out'])
plt.title("Patient Basics test predictions")
plt.show()


In [ ]:
errors = X_test.copy()
errors["actual"] = y_test
errors["predicted"] = prediction
errors = errors[errors["actual"] != errors["predicted"]]
print("Incorrect predictions:", len(errors))
errors.head()


In [ ]:
pd.DataFrame({"train": y_train.value_counts(), "test": y_test.value_counts()})


In [ ]:
feature_names = forest[0].get_feature_names_out()
importance = pd.Series(forest[-1].feature_importances_, index=feature_names)
importance.sort_values().plot.barh(title="Patient forest feature importance")
plt.tight_layout()
plt.show()


## Reading the scores

A fixed random seed repeats the split for the same data order. The three models are practice comparisons. Repeatedly selecting models from the same test scores makes those scores optimistic.


In [ ]:
print("Repeated complete rows:", data.duplicated().sum())
print("Repeated feature rows:", X.duplicated().sum())
# These counts do not tell us whether different rows describe the same person.


In [ ]:
error_counts = errors["actual"].value_counts().reindex(["in", "out"], fill_value=0)
print(pd.DataFrame({"test_rows": y_test.value_counts(), "errors": error_counts}))
